<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 6 </h1></center>
<center><h2> Prompt engineering pour développeurs </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 6 · 15 min · < 0,01 $ · prépare le Lab 2**
> Le lab extrait des demandes de congé ; la démo travaille sur des **tickets RH**, pour ne rien dévoiler.

![](assets/06_messages.png)

In [1]:
import json
import os
from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,
                 base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
print("Prêt")

Prêt


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Le message système </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Le message système fixe **rôle, ton, longueur, limites**, une fois pour toutes.
- Deux questions : une **sur les congés**, une **hors sujet**.

In [2]:
SYSTEM = (
    "Vous êtes l'assistant RH d'une entreprise. Répondez en deux phrases au plus, en vouvoyant. "
    "Si la question ne porte pas sur les congés, répondez exactement : « Je ne traite que les questions de congés. »"
)

for question in ["Je peux poser mes congés en août ?", "Donne-moi une recette de crêpes."]:
    without_system = llm.invoke(question).content
    with_system = llm.invoke([SystemMessage(SYSTEM), HumanMessage(question)]).content
    print("QUESTION :", question)
    print("  sans message système :", len(without_system.split()), "mots —", without_system[:150].replace("\n", " "), "…")
    print("  avec message système :", with_system)
    print()

QUESTION : Je peux poser mes congés en août ?
  sans message système : 68 mots — Oui, vous pouvez généralement poser vos congés en août, mais cela dépend des politiques de votre entreprise et de la disponibilité des ressources. Il  …
  avec message système : Oui, vous pouvez poser vos congés en août, sous réserve de respecter les procédures de demande et de disponibilité. N'hésitez pas à consulter votre responsable pour plus de détails.



QUESTION : Donne-moi une recette de crêpes.
  sans message système : 280 mots — Bien sûr ! Voici une recette simple pour préparer des crêpes :  ### Ingrédients : - 250 g de farine de blé - 3 œufs - 500 ml de lait - 50 g de beurre  …
  avec message système : Je ne traite que les questions de congés.



- Sans : une réponse **longue**, et… **une recette de crêpes**.
- Avec : **2 phrases**, vouvoiement, **refus mot pour mot** (un programme peut le détecter).
- ⚠️ **Pas une barrière de sécurité** : un utilisateur insistant le contourne (Part 5).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Le JSON qui casse </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- On demande du JSON **dans le texte du prompt**, puis on le lit avec `json.loads`.

In [3]:
TICKET = ("Bonjour, mon solde de congés affiché sur le portail est faux, il manque 3 jours "
          "depuis mon retour de congé maternité. Pouvez-vous corriger ? Merci, Julie")

prompt = ("Classez le message. Répondez en JSON avec les clés category "
          "(solde, demande, reclamation, autre) et summary.")
raw_answer = llm.invoke([SystemMessage(prompt), HumanMessage(TICKET)]).content
print(repr(raw_answer))

try:
    json.loads(raw_answer)
except json.JSONDecodeError as error:
    print("\njson.loads échoue :", error)

'```json\n{\n  "category": "reclamation",\n  "summary": "Demande de correction du solde de congés suite à un retour de congé maternité."\n}\n```'

json.loads échoue : Expecting value: line 1 column 1 (char 0)


- JSON correct… **entouré de balises Markdown** → `json.loads` échoue au premier caractère.
- **Demander un format ≠ garantir un format.**

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. La sortie structurée avec Pydantic </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Un schéma Pydantic : **champs, types, valeurs permises, descriptions**.
- `with_structured_output` → l'API **contraint** la génération.

```
class Ticket(BaseModel):
    category: Literal["solde", "demande", ...]   ← valeurs permises
    days: int | None                             ← un entier, ou rien
```

In [4]:
class Ticket(BaseModel):
    category: Literal["solde", "demande", "reclamation", "autre"] = Field(description="Sujet du message")
    summary: str = Field(description="Résumé en une phrase de dix mots au plus")
    days: int | None = Field(description="Nombre de jours mentionnés, ou null")


classifier = llm.with_structured_output(Ticket, include_raw=True)
result = classifier.invoke([HumanMessage(TICKET)])

print("Réponse brute :", result["raw"].content)
print("Objet Python  :", repr(result["parsed"]))
print("Type de days    :", type(result["parsed"].days))

Réponse brute : {"category":"solde","summary":"Correction du solde de congés après congé maternité.","days":3}
Objet Python  : Ticket(category='solde', summary='Correction du solde de congés après congé maternité.', days=3)
Type de days    : <class 'int'>


- Réponse brute : **du JSON pur**, sans balise.
- `days` est **un vrai entier**.
- Les **descriptions guident le choix** (`solde` plutôt que `reclamation`) : un levier aussi fort que le prompt.

- Pydantic **garde aussi la porte côté code**.

In [5]:
try:
    Ticket(category="urgent", summary="Solde faux", days="trois")
except ValidationError as error:
    print(error)

2 validation errors for Ticket
category
  Input should be 'solde', 'demande', 'reclamation' or 'autre' [type=literal_error, input_value='urgent', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
days
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='trois', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


- **Une erreur par champ** : `urgent` hors de la liste, `trois` n'est pas un entier.
- Toute donnée venue d'un modèle ou d'une API **passe par ce contrôle**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Le few-shot </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Une convention **plus facile à montrer qu'à décrire** : l'objet d'un ticket.
- Un exemple = **HumanMessage + AIMessage**.

In [6]:
INSTRUCTION = "Écrivez l'objet d'un ticket RH pour ce message. Répondez par l'objet seul."
EXAMPLES = [
    ("Je voudrais poser du 3 au 7 mars.", "[CONGÉS][DEMANDE] 03/03 → 07/03"),
    ("Mon solde est faux depuis janvier.", "[CONGÉS][SOLDE] écart depuis janvier"),
]
message = "Je me marie le 12 juin, je voudrais mes 4 jours."

zero_shot = llm.invoke([SystemMessage(INSTRUCTION), HumanMessage(message)]).content

few_shot_messages = [SystemMessage(INSTRUCTION)]
for example, subject in EXAMPLES:
    few_shot_messages.append(HumanMessage(example))
    few_shot_messages.append(AIMessage(subject))
few_shot = llm.invoke(few_shot_messages + [HumanMessage(message)]).content

print("zero-shot :", zero_shot)
print("few-shot  :", few_shot)

zero-shot : Demande de congé pour mariage le 12 juin
few-shot  : [CONGÉS][DEMANDE] Mariage le 12 juin - 4 jours


- Zero-shot : un objet **libre**. Few-shot : **[CONGÉS][DEMANDE]**, la forme de l'entreprise.
- Coût : les exemples **repartent à chaque appel**.
- Piège : des exemples trop proches des tests, et on mesure **un modèle qui recopie**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. Répondre à partir d'un contexte </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Ce que le modèle ne sait pas, **on le lui donne** : c'est le principe du RAG.
- Sortie imposée : **`answer` + `sources`**, le format de réponse du projet.

In [7]:
class Answer(BaseModel):
    answer: str = Field(description="Réponse en deux phrases au plus, fondée uniquement sur les extraits")
    sources: list[str] = Field(description="doc_id des extraits utilisés ; liste vide si les extraits ne suffisent pas")


CONTEXT = """[F34154] Le salarié a droit à 4 jours de congé pour son mariage ou la conclusion d'un Pacs.
[F35134] Le salarié a droit à 1 jour de congé pour le mariage de son enfant.
[F2405] Le 1er mai est le seul jour férié obligatoirement chômé."""

answerer = llm.with_structured_output(Answer)
rules = SystemMessage("Répondez uniquement à partir des extraits. S'ils ne suffisent pas, dites-le.")
for question in ["Combien de jours pour mon mariage ?", "Combien de jours pour un déménagement ?"]:
    result = answerer.invoke([rules, HumanMessage(CONTEXT + "\n\nQuestion : " + question)])
    print(question)
    print("  ", result.answer)
    print("   sources :", result.sources)

Combien de jours pour mon mariage ?
   Vous avez droit à 4 jours de congé pour votre mariage ou la conclusion d'un Pacs.
   sources : ['F34154']


Combien de jours pour un déménagement ?
   Les extraits ne mentionnent pas de jours de congé pour un déménagement.
   sources : []


- Mariage : **`sources: ['F34154']`**, et seulement elle.
- Déménagement, absent des extraits : **`sources: []`** → le programme affiche un message de repli.
- La qualité dépend **des extraits** : c'est la Part 8.

**→ Lab 2** : JSON → JSON mode → Pydantic → contexte → few-shot, mesuré à chaque étape.